# Correlation Graph Window Experiment

This notebook plans and stages a controlled test of shorter correlation graph windows: `21`, `63`, and `126` trading days. The current frozen baseline uses `252` trading days. The goal is to test whether more responsive graph structure improves the correlation GNN without replacing or overwriting the existing baseline artifacts.

Research question: does shortening the rolling correlation window improve one-week-ahead realized volatility forecasts by making graph edges more regime-responsive?

Scope rules:

- Keep the existing `252`-day baseline frozen.
- Keep the correlation threshold fixed at the current validation-selected threshold, `config.CORR_THRESHOLD`.
- Keep features, split dates, architecture, loss, and random seed fixed across windows.
- Select any preferred window using validation results only.
- Treat test metrics as final evaluation after the window choice is fixed.
- Register every trained window model with explicit `graph_version` provenance.

## Experiment Plan

1. Build versioned correlation edge files for `21`, `63`, and `126` day lookbacks.
2. Compare graph diagnostics against the existing `252` day graph: edge count, density, mean degree, and week-to-week edge turnover.
3. Train one correlation GNN per window using the same macro feature tensor and tuned macro correlation GNN architecture.
4. Save validation and test predictions with window-specific artifact names.
5. Compare validation MSE, Rank IC, prediction spread, and calibration slope.
6. Choose the preferred lookback using validation only.
7. Evaluate the selected window on test metrics, portfolio metrics, and significance tests in the downstream evaluation notebooks.

This notebook is intentionally separate from `04_gnn_models.ipynb` and `04b_gnn_hparam_search.ipynb` so the graph-window experiment is auditable and does not silently alter the main baseline.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from src.graphs import build_correlation_graph
from src.models import GNNModelV2
from src.train import predict_gnn_split, set_seeds, train_gnn

pd.set_option('display.max_columns', 120)

WINDOWS = [21, 63, 126]
REFERENCE_WINDOW = config.CORR_LOOKBACK_DAYS
THRESHOLD = config.CORR_THRESHOLD
FEATURE_VERSION = 'stock_features_plus_regime_v1'
FEATURE_PATH = Path(config.DATA_FEATURES_DIR) / 'features_macro.parquet'
META_PATH = Path(config.DATA_FEATURES_DIR) / 'features_macro_meta.json'
WINDOW_EDGE_DIR = Path(config.DATA_GRAPHS_DIR) / 'corr_edges_window'
RESULTS_DIR = Path(config.DATA_RESULTS_DIR)
CHECKPOINT_DIR = Path(config.CHECKPOINTS_DIR)

# Graph diagnostics have been reviewed; train all window models.
RUN_TRAINING = True

WINDOW_EDGE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('Windows to test:', WINDOWS)
print('Reference window:', REFERENCE_WINDOW)
print('Threshold:', THRESHOLD)
print('Feature version:', FEATURE_VERSION)
print('Window edge dir:', WINDOW_EDGE_DIR)

Windows to test: [21, 63, 126]
Reference window: 252
Threshold: 0.3
Feature version: stock_features_plus_regime_v1
Window edge dir: C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\graphs\corr_edges_window


## Artifact Naming

New graph files are written under `data/graphs/corr_edges_window/` instead of the default `data/graphs/corr_edges/`. This prevents accidental overwrite of the official `252` day edge files.

Naming pattern:

- Edges: `corr_edges_{split}_t0p3_w021.parquet`, `w063`, `w126`
- Checkpoints: `data/results/checkpoints/gnn_corr_macro_w021_best.pt`, etc.
- Val losses: `data/results/gnn_corr_macro_w021_val_loss.json`, etc.
- Predictions: `data/results/gnn_corr_macro_w021_val_preds.parquet` and `data/results/test_preds_gnn_corr_macro_w021.parquet`
- Registry `graph_version`: `correlation_threshold_0.3_lookback_21`, etc.

In [2]:
def threshold_tag(threshold: float) -> str:
    return 't' + f'{threshold:.1f}'.replace('.', 'p')


def window_tag(window: int) -> str:
    return f'w{window:03d}'


def edge_path(split: str, window: int, threshold: float = THRESHOLD) -> Path:
    return WINDOW_EDGE_DIR / f'corr_edges_{split}_{threshold_tag(threshold)}_{window_tag(window)}.parquet'


def graph_version(window: int, threshold: float = THRESHOLD) -> str:
    return f'correlation_threshold_{threshold}_lookback_{window}'


artifact_plan = pd.DataFrame([
    {
        'window': w,
        'graph_version': graph_version(w),
        'train_edges': edge_path('train', w).as_posix(),
        'val_edges': edge_path('val', w).as_posix(),
        'test_edges': edge_path('test', w).as_posix(),
        'checkpoint': (CHECKPOINT_DIR / f'gnn_corr_macro_{window_tag(w)}_best.pt').as_posix(),
        'test_predictions': (RESULTS_DIR / f'test_preds_gnn_corr_macro_{window_tag(w)}.parquet').as_posix(),
    }
    for w in WINDOWS
])
artifact_plan

,window,graph_version,train_edges,val_edges,test_edges,checkpoint,test_predictions
0,21,correlation_threshold_0.3_lookback_21,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...
1,63,correlation_threshold_0.3_lookback_63,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...
2,126,correlation_threshold_0.3_lookback_126,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...


## Load Shared Data

The graph-window comparison should use the same macro feature tensor as the current best macro correlation GNN. This isolates the graph lookback window as the experimental variable.

In [3]:
target = pd.read_parquet(Path(config.DATA_FEATURES_DIR) / 'target.parquet')
splits = pd.read_parquet(Path(config.DATA_FEATURES_DIR) / 'splits.parquet')
log_returns = pd.read_parquet(Path(config.DATA_RAW_DIR) / 'log_returns.parquet')

target.index = pd.to_datetime(target.index)
splits['week'] = pd.to_datetime(splits['week'])
log_returns.index = pd.to_datetime(log_returns.index)

tickers = target.columns.tolist()
log_returns = log_returns[tickers]

with open(META_PATH, encoding='utf-8') as fh:
    meta = json.load(fh)
feature_names = meta['feature_names']

feat_df = pd.read_parquet(FEATURE_PATH)
feat_df['week'] = pd.to_datetime(feat_df['week'])
feat_df = feat_df[feat_df['week'].isin(target.index)].sort_values(['week', 'ticker']).reset_index(drop=True)

n_weeks = target.shape[0]
n_stocks = target.shape[1]
n_feats = len(feature_names)
features_3d = feat_df[feature_names].to_numpy(dtype=float).reshape(n_weeks, n_stocks, n_feats)
target_arr = target.to_numpy(dtype=float)
week_index = pd.DatetimeIndex(target.index)

assert features_3d.shape == (n_weeks, n_stocks, n_feats)
assert target_arr.shape == (n_weeks, n_stocks)
assert list(feat_df.loc[feat_df['week'].eq(week_index[0]), 'ticker']) == tickers

pd.DataFrame({
    'item': ['features_3d', 'target_arr', 'weeks', 'stocks', 'features'],
    'value': [str(features_3d.shape), str(target_arr.shape), n_weeks, n_stocks, n_feats],
})

,item,value
0,features_3d,"(572, 465, 19)"
1,target_arr,"(572, 465)"
2,weeks,572
3,stocks,465
4,features,19


## Precompute Windowed Graphs

The existing `src.graphs.build_correlation_graph` already accepts a `lookback` parameter. The only architectural change needed is versioned storage. Re-running this cell is safe because existing files are skipped.

In [4]:
def precompute_window_graphs(
    log_returns: pd.DataFrame,
    splits: pd.DataFrame,
    windows: list[int],
    threshold: float = THRESHOLD,
) -> pd.DataFrame:
    rows = []
    for window in windows:
        for split_name in ('train', 'val', 'test'):
            out_path = edge_path(split_name, window, threshold)
            split_weeks = sorted(splits.loc[splits['split'].eq(split_name), 'week'].tolist())
            if out_path.exists():
                df_existing = pd.read_parquet(out_path)
                rows.append({
                    'window': window,
                    'split': split_name,
                    'path': out_path.as_posix(),
                    'status': 'exists',
                    'rows': len(df_existing),
                    'weeks': len(split_weeks),
                })
                continue

            frames = []
            for week in split_weeks:
                friday = week + pd.Timedelta(days=4)
                edge_index = build_correlation_graph(log_returns, friday, lookback=window, threshold=threshold)
                if edge_index.shape[1] == 0:
                    continue
                n_edges = edge_index.shape[1]
                frames.append(pd.DataFrame({
                    'week': np.full(n_edges, str(week.date())),
                    'src': edge_index[0].numpy().astype(np.int32),
                    'dst': edge_index[1].numpy().astype(np.int32),
                }))

            out_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame({
                'week': pd.Series(dtype=str),
                'src': pd.Series(dtype='int32'),
                'dst': pd.Series(dtype='int32'),
            })
            out_df.to_parquet(out_path, index=False)
            rows.append({
                'window': window,
                'split': split_name,
                'path': out_path.as_posix(),
                'status': 'created',
                'rows': len(out_df),
                'weeks': len(split_weeks),
            })
    return pd.DataFrame(rows)


graph_build_summary = precompute_window_graphs(log_returns, splits, WINDOWS)
graph_build_summary

C:\Users\Rylan Wade\AppData\Local\Temp\ipykernel_20696\498149812.py:27: UserWarning: build_correlation_graph: fewer than requested rows available — correlation estimated on a shorter window (expected at dataset start).
  edge_index = build_correlation_graph(log_returns, friday, lookback=window, threshold=threshold)


,window,split,path,status,rows,weeks
0,21,train,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,49738610,417
1,21,val,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,5720886,52
2,21,test,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,10004560,103
3,63,train,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,46228942,417
4,63,val,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,5171220,52
5,63,test,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,7935524,103
6,126,train,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,46783290,417
7,126,val,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,5771318,52
8,126,test,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,created,7937688,103


## Graph Diagnostics Before Training

Short windows may be more responsive, but they can also create noisier and denser or sparser graphs. Review these diagnostics before training.

In [5]:
def load_window_graphs(window: int, split_name: str, threshold: float = THRESHOLD) -> dict[pd.Timestamp, torch.LongTensor]:
    path = edge_path(split_name, window, threshold)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    graphs = {}
    for week_str, group in df.groupby('week', sort=True):
        graphs[pd.Timestamp(week_str)] = torch.tensor(
            np.vstack([group['src'].to_numpy(np.int64), group['dst'].to_numpy(np.int64)]),
            dtype=torch.long,
        )
    return graphs


def graph_diagnostics(window: int) -> pd.DataFrame:
    rows = []
    possible_directed = n_stocks * (n_stocks - 1)
    for split_name in ('train', 'val', 'test'):
        path = edge_path(split_name, window)
        df = pd.read_parquet(path)
        split_weeks = sorted(splits.loc[splits['split'].eq(split_name), 'week'].tolist())
        for week in split_weeks:
            week_key = str(week.date())
            week_edges = df.loc[df['week'].eq(week_key)]
            directed_edges = len(week_edges)
            rows.append({
                'window': window,
                'split': split_name,
                'week': week,
                'directed_edges': directed_edges,
                'unique_edges': directed_edges / 2,
                'density': directed_edges / possible_directed,
                'mean_degree': directed_edges / n_stocks,
            })
    return pd.DataFrame(rows)


density = pd.concat([graph_diagnostics(w) for w in WINDOWS], ignore_index=True)
density_summary = (
    density.groupby(['window', 'split'])
    .agg(
        mean_density=('density', 'mean'),
        median_density=('density', 'median'),
        mean_degree=('mean_degree', 'mean'),
        min_edges=('directed_edges', 'min'),
        max_edges=('directed_edges', 'max'),
    )
    .reset_index()
)
density_summary

,window,split,mean_density,median_density,mean_degree,min_edges,max_edges
0,21,test,0.450184,0.412143,208.885270,60894,191330
1,21,train,0.552824,0.519559,256.510198,48720,212840
2,21,val,0.509905,0.490832,236.595782,56608,169466
3,63,test,0.357082,0.292306,165.685854,42102,170754
4,63,train,0.513815,0.498248,238.410263,19720,211256
5,63,val,0.460913,0.458097,213.863524,48134,170120
6,126,test,0.357179,0.274518,165.731037,39238,145006
7,126,train,0.519977,0.511170,241.269127,18136,207050
8,126,val,0.514400,0.466500,238.681472,59414,177348


In [6]:
def edge_turnover(window: int, split_name: str = 'val') -> pd.DataFrame:
    path = edge_path(split_name, window)
    df = pd.read_parquet(path)
    weeks = sorted(splits.loc[splits['split'].eq(split_name), 'week'].tolist())
    rows = []
    prev_edges = None
    for week in weeks:
        week_key = str(week.date())
        g = df.loc[df['week'].eq(week_key), ['src', 'dst']]
        current = set(map(tuple, g.to_numpy(dtype=int)))
        if prev_edges is not None:
            union = len(current | prev_edges)
            changed = len(current ^ prev_edges)
            rows.append({
                'window': window,
                'split': split_name,
                'week': week,
                'edge_turnover': changed / union if union else np.nan,
            })
        prev_edges = current
    return pd.DataFrame(rows)


turnover_summary = (
    pd.concat([edge_turnover(w, 'val') for w in WINDOWS], ignore_index=True)
    .groupby('window')
    .agg(mean_val_edge_turnover=('edge_turnover', 'mean'), median_val_edge_turnover=('edge_turnover', 'median'))
    .reset_index()
)
turnover_summary

,window,mean_val_edge_turnover,median_val_edge_turnover
0,21,0.336659,0.336900
1,63,0.164139,0.148316
2,126,0.090887,0.083468


## Optional Training Cell

Set `RUN_TRAINING = True` only after reviewing graph diagnostics. This trains one macro-feature correlation GNN per window using the same tuned architecture style as the current macro hparam winner. The graph lookback is the only intended variable.

In [7]:
def load_macro_best_config() -> dict:
    path = RESULTS_DIR / 'gnn_corr_macro_hparam_search_results.json'
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        return data.get('best_config', {})
    return {
        'lr': config.GNN_LEARNING_RATE,
        'hidden_dim': config.HIDDEN_DIM,
        'dropout': config.DROPOUT,
        'batch_norm': False,
        'num_layers': config.GNN_NUM_LAYERS,
    }


best_cfg = load_macro_best_config()
best_cfg

{'lr': 0.0003,
 'hidden_dim': 256,
 'dropout': 0.3,
 'batch_norm': True,
 'num_layers': 3}

In [12]:
RUN_TRAINING = True

In [13]:
# Resume-safe training cell: this can be run after an interruption without rerunning the notebook from the top.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from src.models import GNNModelV2
from src.train import predict_gnn_split, set_seeds, train_gnn

WINDOWS = globals().get('WINDOWS', [21, 63, 126])
THRESHOLD = globals().get('THRESHOLD', config.CORR_THRESHOLD)
FEATURE_VERSION = globals().get('FEATURE_VERSION', 'stock_features_plus_regime_v1')
FEATURE_PATH = globals().get('FEATURE_PATH', Path(config.DATA_FEATURES_DIR) / 'features_macro.parquet')
META_PATH = globals().get('META_PATH', Path(config.DATA_FEATURES_DIR) / 'features_macro_meta.json')
WINDOW_EDGE_DIR = globals().get('WINDOW_EDGE_DIR', Path(config.DATA_GRAPHS_DIR) / 'corr_edges_window')
RESULTS_DIR = globals().get('RESULTS_DIR', Path(config.DATA_RESULTS_DIR))
CHECKPOINT_DIR = globals().get('CHECKPOINT_DIR', Path(config.CHECKPOINTS_DIR))
RUN_TRAINING = globals().get('RUN_TRAINING', True)

WINDOW_EDGE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def threshold_tag(threshold: float) -> str:
    return 't' + f'{threshold:.1f}'.replace('.', 'p')


def window_tag(window: int) -> str:
    return f'w{window:03d}'


def edge_path(split: str, window: int, threshold: float = THRESHOLD) -> Path:
    return WINDOW_EDGE_DIR / f'corr_edges_{split}_{threshold_tag(threshold)}_{window_tag(window)}.parquet'


def graph_version(window: int, threshold: float = THRESHOLD) -> str:
    return f'correlation_threshold_{threshold}_lookback_{window}'


def load_window_graphs(window: int, split_name: str, threshold: float = THRESHOLD) -> dict[pd.Timestamp, torch.LongTensor]:
    path = edge_path(split_name, window, threshold)
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run the graph precompute cell first.')
    df = pd.read_parquet(path)
    graphs = {}
    for week_str, group in df.groupby('week', sort=True):
        graphs[pd.Timestamp(week_str)] = torch.tensor(
            np.vstack([group['src'].to_numpy(np.int64), group['dst'].to_numpy(np.int64)]),
            dtype=torch.long,
        )
    return graphs


def load_macro_best_config() -> dict:
    path = RESULTS_DIR / 'gnn_corr_macro_hparam_search_results.json'
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        return data.get('best_config', {})
    return {
        'lr': config.GNN_LEARNING_RATE,
        'hidden_dim': config.HIDDEN_DIM,
        'dropout': config.DROPOUT,
        'batch_norm': False,
        'num_layers': config.GNN_NUM_LAYERS,
    }


if 'features_3d' not in globals() or 'target_arr' not in globals() or 'week_index' not in globals():
    print('Reloading shared macro feature tensor and target arrays...')
    target = pd.read_parquet(Path(config.DATA_FEATURES_DIR) / 'target.parquet')
    splits = pd.read_parquet(Path(config.DATA_FEATURES_DIR) / 'splits.parquet')
    target.index = pd.to_datetime(target.index)
    splits['week'] = pd.to_datetime(splits['week'])
    tickers = target.columns.tolist()

    with open(META_PATH, encoding='utf-8') as fh:
        meta = json.load(fh)
    feature_names = meta['feature_names']
    feat_df = pd.read_parquet(FEATURE_PATH)
    feat_df['week'] = pd.to_datetime(feat_df['week'])
    feat_df = feat_df[feat_df['week'].isin(target.index)].sort_values(['week', 'ticker']).reset_index(drop=True)

    n_weeks = target.shape[0]
    n_stocks = target.shape[1]
    n_feats = len(feature_names)
    features_3d = feat_df[feature_names].to_numpy(dtype=float).reshape(n_weeks, n_stocks, n_feats)
    target_arr = target.to_numpy(dtype=float)
    week_index = pd.DatetimeIndex(target.index)

best_cfg = load_macro_best_config()
training_records = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('RUN_TRAINING:', RUN_TRAINING)
print('Windows:', WINDOWS)
print('Best config:', best_cfg)

if RUN_TRAINING:
    for window in WINDOWS:
        tag = f'corr_macro_{window_tag(window)}'
        print(f'\n=== {tag} ===')
        graphs = {}
        for split_name in ('train', 'val', 'test'):
            graphs.update(load_window_graphs(window, split_name))
        edge_fn = lambda week, g=graphs: g.get(week, torch.zeros(2, 0, dtype=torch.long))

        set_seeds()
        model = GNNModelV2(
            in_channels=features_3d.shape[2],
            hidden_dim=int(best_cfg.get('hidden_dim', config.HIDDEN_DIM)),
            dropout=float(best_cfg.get('dropout', config.DROPOUT)),
            num_layers=int(best_cfg.get('num_layers', config.GNN_NUM_LAYERS)),
            batch_norm=bool(best_cfg.get('batch_norm', False)),
        ).to(device)

        ckpt_path = CHECKPOINT_DIR / f'gnn_{tag}_best.pt'
        loss_path = RESULTS_DIR / f'gnn_{tag}_val_loss.json'
        if ckpt_path.exists() and loss_path.exists():
            print(f'{tag}: checkpoint and loss exist, loading instead of retraining.')
            model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
            loss_history = json.loads(loss_path.read_text(encoding='utf-8'))['val_loss']
        else:
            model, loss_history = train_gnn(
                model=model,
                features=features_3d,
                target=target_arr,
                week_index=week_index,
                edge_index_fn=edge_fn,
                splits=splits,
                graph_type=tag,
                device=device,
                max_epochs=config.GNN_MAX_EPOCHS,
                lr=float(best_cfg.get('lr', config.GNN_LEARNING_RATE)),
                patience=config.GNN_HPARAM_PATIENCE,
                save_periodic=False,
            )

        val_path = RESULTS_DIR / f'gnn_{tag}_val_preds.parquet'
        test_path = RESULTS_DIR / f'test_preds_gnn_{tag}.parquet'
        if val_path.exists() and test_path.exists():
            print(f'{tag}: validation/test predictions already exist, keeping them.')
        else:
            val_preds = predict_gnn_split(model, features_3d, week_index, edge_fn, splits, tickers, device, split_name='val')
            test_preds = predict_gnn_split(model, features_3d, week_index, edge_fn, splits, tickers, device, split_name='test')
            val_preds.to_parquet(val_path)
            test_preds.to_parquet(test_path)
            print(f'{tag}: saved predictions.')

        training_records.append({
            'window': window,
            'graph_version': graph_version(window),
            'best_val_mse': min(loss_history),
            'n_epochs': len(loss_history),
            'checkpoint_path': ckpt_path.as_posix(),
            'validation_prediction_path': val_path.as_posix(),
            'prediction_path': test_path.as_posix(),
        })
else:
    print('RUN_TRAINING is False. Set it to True to train all window models.')

pd.DataFrame(training_records).sort_values('best_val_mse') if training_records else pd.DataFrame()

Device: cuda
RUN_TRAINING: True
Windows: [21, 63, 126]
Best config: {'lr': 0.0003, 'hidden_dim': 256, 'dropout': 0.3, 'batch_norm': True, 'num_layers': 3}

=== corr_macro_w021 ===


c:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\models.py:350: UserWarning: GNNModelV2.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.076677  val=0.027090
Epoch   2  train=0.046495  val=0.026851
Epoch   3  train=0.042945  val=0.026595
Epoch   4  train=0.041205  val=0.026665
Epoch   5  train=0.039969  val=0.026527
Epoch   6  train=0.039288  val=0.026424
Epoch   7  train=0.038555  val=0.025749
Epoch   8  train=0.037590  val=0.024590
Epoch   9  train=0.036586  val=0.023541
Epoch  10  train=0.035655  val=0.022390
Epoch  11  train=0.034687  val=0.021942
Epoch  12  train=0.033534  val=0.021870
Epoch  13  train=0.032790  val=0.021737
Epoch  14  train=0.032212  val=0.022248
Epoch  15  train=0.031807  val=0.022224
Epoch  16  train=0.030956  val=0.022076
Epoch  17  train=0.030913  val=0.021823
Epoch  18  train=0.029940  val=0.021461
Epoch  19  train=0.029149  val=0.021361
Epoch  20  train=0.028731  val=0.021158
Epoch  21  train=0.027812  val=0.021662
Epoch  22  train=0.027010  val=0.021119
Epoch  23  train=0.026424  val=0.020947
Epoch  24  train=0.025877  val=0.020743
Epoch  25  train=0.025136  val=0.020675


c:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\models.py:350: UserWarning: GNNModelV2.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.077017  val=0.027299
Epoch   2  train=0.046450  val=0.027924
Epoch   3  train=0.042875  val=0.027790
Epoch   4  train=0.041276  val=0.027988
Epoch   5  train=0.040056  val=0.027841
Epoch   6  train=0.039409  val=0.028037
Epoch   7  train=0.038681  val=0.027548
Epoch   8  train=0.037766  val=0.025378
Epoch   9  train=0.037113  val=0.024615
Epoch  10  train=0.036529  val=0.023861
Epoch  11  train=0.035906  val=0.023270
Epoch  12  train=0.035201  val=0.022695
Epoch  13  train=0.034625  val=0.021960
Epoch  14  train=0.034067  val=0.021863
Epoch  15  train=0.033526  val=0.021612
Epoch  16  train=0.032923  val=0.021259
Epoch  17  train=0.032732  val=0.021197
Epoch  18  train=0.032043  val=0.021129
Epoch  19  train=0.031528  val=0.021106
Epoch  20  train=0.031247  val=0.021146
Epoch  21  train=0.030735  val=0.020975
Epoch  22  train=0.030305  val=0.021044
Epoch  23  train=0.029902  val=0.020964
Epoch  24  train=0.029576  val=0.021190
Epoch  25  train=0.029142  val=0.021002


c:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\.venv\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\src\models.py:350: UserWarning: GNNModelV2.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.077198  val=0.026994
Epoch   2  train=0.046402  val=0.027123
Epoch   3  train=0.042821  val=0.027075
Epoch   4  train=0.041221  val=0.027552
Epoch   5  train=0.039953  val=0.027867
Epoch   6  train=0.039403  val=0.027682
Epoch   7  train=0.038703  val=0.027069
Epoch   8  train=0.038027  val=0.025048
Epoch   9  train=0.037372  val=0.024351
Epoch  10  train=0.036855  val=0.023617
Epoch  11  train=0.036298  val=0.023131
Epoch  12  train=0.035672  val=0.022745
Epoch  13  train=0.035129  val=0.022070
Epoch  14  train=0.034570  val=0.021885
Epoch  15  train=0.034030  val=0.021687
Epoch  16  train=0.033358  val=0.021351
Epoch  17  train=0.033152  val=0.021352
Epoch  18  train=0.032504  val=0.021271
Epoch  19  train=0.031864  val=0.021199
Epoch  20  train=0.031548  val=0.021167
Epoch  21  train=0.030849  val=0.021120
Epoch  22  train=0.030385  val=0.021004
Epoch  23  train=0.029901  val=0.020892
Epoch  24  train=0.029460  val=0.020993
Epoch  25  train=0.028929  val=0.020826


,window,graph_version,best_val_mse,n_epochs,checkpoint_path,validation_prediction_path,prediction_path
0,21,correlation_threshold_0.3_lookback_21,0.019857,51,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...
2,126,correlation_threshold_0.3_lookback_126,0.019911,53,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...
1,63,correlation_threshold_0.3_lookback_63,0.020044,53,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...,C:/Users/Rylan Wade/Desktop/FinancialAnalytics...


## Validation Comparison

Use validation metrics to choose the lookback. Test results should be read only after the preferred window is selected.

In [9]:
def rank_ic_by_week(preds: pd.DataFrame, actuals: pd.DataFrame) -> float:
    vals = []
    for week in preds.index:
        p = preds.loc[week]
        a = actuals.loc[week]
        valid = p.notna() & a.notna()
        if valid.sum() >= 3:
            vals.append(p[valid].rank().corr(a[valid].rank()))
    return float(np.nanmean(vals)) if vals else np.nan


def calibration_slope(preds: pd.DataFrame, actuals: pd.DataFrame) -> float:
    p = preds.to_numpy().ravel()
    a = actuals.to_numpy().ravel()
    valid = np.isfinite(p) & np.isfinite(a)
    if valid.sum() < 3:
        return np.nan
    return float(np.polyfit(p[valid], a[valid], 1)[0])


def prediction_spread(preds: pd.DataFrame) -> float:
    return float((preds.quantile(0.90, axis=1) - preds.quantile(0.10, axis=1)).mean())


def validation_metrics_for_saved_predictions() -> pd.DataFrame:
    val_actual = target.loc[splits.loc[splits['split'].eq('val'), 'week'].sort_values()]
    rows = []
    for window in WINDOWS:
        tag = f'corr_macro_{window_tag(window)}'
        path = RESULTS_DIR / f'gnn_{tag}_val_preds.parquet'
        if not path.exists():
            rows.append({'window': window, 'status': 'missing_predictions'})
            continue
        preds = pd.read_parquet(path)
        preds.index = pd.to_datetime(preds.index)
        actual = val_actual.loc[preds.index, preds.columns]
        err = preds.to_numpy() - actual.to_numpy()
        rows.append({
            'window': window,
            'status': 'ok',
            'val_mse': float(np.nanmean(err ** 2)),
            'val_mae': float(np.nanmean(np.abs(err))),
            'val_rank_ic': rank_ic_by_week(preds, actual),
            'val_calibration_slope': calibration_slope(preds, actual),
            'val_prediction_spread': prediction_spread(preds),
        })
    return pd.DataFrame(rows)


validation_window_summary = validation_metrics_for_saved_predictions()
validation_window_summary

,window,status
0,21,missing_predictions
1,63,missing_predictions
2,126,missing_predictions


## Validation-Only Selection Rule

Pick a window before reading test results. Recommended rule:

1. Primary criterion: lowest validation MSE.
2. Tie-breaker 1: materially better validation Rank IC.
3. Tie-breaker 2: calibration slope closer to `1.0`.
4. Reject a window if graph diagnostics show extreme instability or density collapse.

Once selected, add the selected model's prediction file to the evaluation and significance scripts/notebooks as a new model row. Do not replace `GNN-Correlation + Macro Tuned` unless the report explicitly switches the final roster.

In [10]:
def draft_registry_rows() -> pd.DataFrame:
    base_hparams = {
        'random_seed': config.RANDOM_SEED,
        'architecture': 'GNNModelV2',
        'hidden_dim': int(best_cfg.get('hidden_dim', config.HIDDEN_DIM)),
        'num_layers': int(best_cfg.get('num_layers', config.GNN_NUM_LAYERS)),
        'dropout': float(best_cfg.get('dropout', config.DROPOUT)),
        'batch_norm': bool(best_cfg.get('batch_norm', False)),
        'learning_rate': float(best_cfg.get('lr', config.GNN_LEARNING_RATE)),
        'corr_threshold': THRESHOLD,
        'max_epochs': config.GNN_MAX_EPOCHS,
        'early_stop_patience': config.GNN_HPARAM_PATIENCE,
    }
    rows = []
    for window in WINDOWS:
        tag = f'corr_macro_{window_tag(window)}'
        hparams = {**base_hparams, 'corr_lookback_days': window}
        rows.append({
            'experiment_id': f'window_gnn_correlation_macro_{window}',
            'model_name': f'GNN-Correlation + Macro {window}d',
            'model_family': 'GNN',
            'graph_type': 'correlation',
            'loss_type': 'mse',
            'feature_version': FEATURE_VERSION,
            'graph_version': graph_version(window),
            'checkpoint_path': f'data/results/checkpoints/gnn_{tag}_best.pt',
            'train_split': f'train <= {config.TRAIN_END}',
            'val_split': f'val <= {config.VAL_END}',
            'test_split': f'test <= {config.TEST_END}',
            'hyperparameters': json.dumps(hparams, sort_keys=True, separators=(',', ':')),
            'validation_metrics_path': f'data/results/gnn_{tag}_val_loss.json',
            'test_metrics_path': '',
            'portfolio_metrics_path': '',
            'prediction_path': f'data/results/test_preds_gnn_{tag}.parquet',
            'validation_prediction_path': f'data/results/gnn_{tag}_val_preds.parquet',
            'notes': 'Correlation graph lookback-window experiment. Window selected using validation only.',
        })
    return pd.DataFrame(rows)


registry_draft = draft_registry_rows()
registry_draft

,experiment_id,model_name,model_family,graph_type,loss_type,feature_version,graph_version,checkpoint_path,train_split,val_split,test_split,hyperparameters,validation_metrics_path,test_metrics_path,portfolio_metrics_path,prediction_path,validation_prediction_path,notes
0,window_gnn_correlation_macro_21,GNN-Correlation + Macro 21d,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_21,data/results/checkpoints/gnn_corr_macro_w021_b...,train <= 2022-12-31,val <= 2023-12-31,test <= 2025-12-31,"{""architecture"":""GNNModelV2"",""batch_norm"":true...",data/results/gnn_corr_macro_w021_val_loss.json,,,data/results/test_preds_gnn_corr_macro_w021.pa...,data/results/gnn_corr_macro_w021_val_preds.par...,Correlation graph lookback-window experiment. ...
1,window_gnn_correlation_macro_63,GNN-Correlation + Macro 63d,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_63,data/results/checkpoints/gnn_corr_macro_w063_b...,train <= 2022-12-31,val <= 2023-12-31,test <= 2025-12-31,"{""architecture"":""GNNModelV2"",""batch_norm"":true...",data/results/gnn_corr_macro_w063_val_loss.json,,,data/results/test_preds_gnn_corr_macro_w063.pa...,data/results/gnn_corr_macro_w063_val_preds.par...,Correlation graph lookback-window experiment. ...
2,window_gnn_correlation_macro_126,GNN-Correlation + Macro 126d,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_126,data/results/checkpoints/gnn_corr_macro_w126_b...,train <= 2022-12-31,val <= 2023-12-31,test <= 2025-12-31,"{""architecture"":""GNNModelV2"",""batch_norm"":true...",data/results/gnn_corr_macro_w126_val_loss.json,,,data/results/test_preds_gnn_corr_macro_w126.pa...,data/results/gnn_corr_macro_w126_val_preds.par...,Correlation graph lookback-window experiment. ...


## Downstream Evaluation Checklist

After the window models are trained and a validation-selected window is chosen:

- Add the selected prediction file to `src/evaluate.py` or the relevant evaluation notebook model map.
- Regenerate ML metrics, Rank IC, calibration, and prediction-spread tables.
- Add the selected model to portfolio evaluation only after forecasting metrics look credible.
- Add the selected model to `scripts/run_significance.py` for DM tests using weekly MSE series.
- Update `data/results/experiment_registry.csv` with the final row, not just this draft.
- Update the results handoff document only after final artifacts are regenerated.

Recommended report language if shorter windows help: the original `252` day graph was stable but slow to adapt; shorter windows improved regime responsiveness. If they do not help: the `252` day graph's stability was more valuable than short-window responsiveness in this sample.